# CLM (Contrastive Language Model) - Function Calling Demo

Demo chạy CLM trên function calling task

**Model:** CLM-v0.1-8B
**Dataset:** Multi-turn function calling dataset

## 1. Install Dependencies

In [ ]:
# 1. Check GPU availability (Google Colab requires T4 GPU, L4 or A100 runtime)
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    !nvidia-smi
else:
    print("\n" + "!" * 70)
    print("❌ CẢNH BÁO: BẠN ĐANG CHẠY TRÊN CPU, CHƯA BẬT GPU!")
    print("vLLM và mô hình 8B bắt buộc phải có GPU để chạy.")
    print("👉 HÃY LÀM THEO BƯỚC SAU TRÊN COLAB:")
    print("   1. Menu Runtime (Thời gian chạy) -> Change runtime type (Thay đổi loại thời gian chạy)")
    print("   2. Hardware accelerator (Bộ tăng tốc phần cứng) -> Chọn 'T4 GPU'")
    print("   3. Bấm 'Save' (Lưu)")
    print("!" * 70 + "\n")

# 2. Gỡ bỏ torchaudio bị lệch CUDA version (Qwen3 & CLM chỉ xử lý text, hoàn toàn không cần torchaudio)
!pip uninstall -y torchaudio

# 3. Install CLM, vLLM, bitsandbytes and dependencies
!pip install contrastive-lm vllm transformers accelerate bitsandbytes -q
!pip install psmisc -q  # Provides fuser utility to clean up ports


## 2. Setup Configuration

In [ ]:
import os
import json
import time
import subprocess
import urllib.request
from typing import List, Dict, Any, Optional

# Configuration
VLLM_PORT = 8090
CLM_PORT = 8700

# Trên GPU Tesla T4 (15GB VRAM), model 8B FP16 chiếm 16GB sẽ bị Out-Of-Memory (OOM).
# Sử dụng Qwen/Qwen3-8B-AWQ (4-bit, chỉ chiếm ~6GB VRAM) để chạy mượt mà trên T4.
# Nếu bạn dùng Colab Pro với GPU L4 (24GB) hoặc A100 (40GB), có thể đổi lại thành "Qwen/Qwen3-8B".
VLLM_MODEL = "Qwen/Qwen3-8B-AWQ"
VLLM_URL = f"http://127.0.0.1:{VLLM_PORT}"
CLM_URL = f"http://127.0.0.1:{CLM_PORT}"

os.environ["CLM_BASE_URL"] = CLM_URL

print("Configuration loaded!")
print(f"Selected Model: {VLLM_MODEL}")
print(f"vLLM URL (Embedder): {VLLM_URL}")
print(f"CLM URL (System One API): {CLM_URL}")


## 3. Start vLLM Server (Background)

In [ ]:
import sys
import os
import time
import subprocess
import urllib.request
import torch

# 1. Bắt buộc phải có GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "\n" + "="*70 + "\n" +
        "❌ LỖI: Runtime Colab hiện tại là CPU (Không tìm thấy GPU)!\n\n" +
        "vLLM không thể khởi động trên CPU và báo lỗi: Failed to infer device type.\n" +
        "👉 Cách sửa trên Google Colab:\n" +
        "   1. Chọn menu: Runtime (Thời gian chạy) -> Change runtime type (Thay đổi loại thời gian chạy)\n" +
        "   2. Mục Hardware accelerator (Bộ tăng tốc phần cứng): chọn \x27T4 GPU\x27\n" +
        "   3. Bấm \x27Save\x27 (Lưu) rồi chạy lại từ Cell 1.\n" +
        "="*70
    )

# 2. Cấu hình tương thích cho Tesla T4 (Turing SM 7.5 không hỗ trợ FlashAttention-2)
# Tắt vLLM V1 engine và chuyển sang backend TORCH_SDPA tương thích 100% với T4
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ATTENTION_BACKEND"] = "TORCH_SDPA"

# 3. Gỡ bỏ torchaudio để tránh xung đột phiên bản CUDA giữa PyTorch và TorchAudio
!pip uninstall -y torchaudio 2>/dev/null || true

# 4. Kill existing zombie processes on ports 8090 and 8700
!fuser -k 8090/tcp 2>/dev/null || true
!fuser -k 8700/tcp 2>/dev/null || true
time.sleep(2)

def start_services():
    """Start vLLM (embedding backend) and CLM API server in background with robust logging"""
    
    vllm_env = os.environ.copy()
    vllm_env["VLLM_USE_V1"] = "0"
    vllm_env["VLLM_ATTENTION_BACKEND"] = "TORCH_SDPA"
    
    # vLLM serve flags optimized for Google Colab (T4 / L4 / A100)
    vllm_cmd = [
        "vllm", "serve", VLLM_MODEL,
        "--served-model-name", "qwen3-8b",
        "--runner", "pooling",
        "--max-model-len", "2048",
        "--port", str(VLLM_PORT),
        "--tensor-parallel-size", "1",
        "--enforce-eager",              # Saves ~1.5GB VRAM on T4 (disables CUDA graph capture)
        "--gpu-memory-utilization", "0.90",
    ]
    
    # Chỉ thêm --dtype float16 nếu dùng model FP16 gốc, bản AWQ tự động nhận kiểu dữ liệu
    if "AWQ" not in VLLM_MODEL:
        vllm_cmd.extend(["--dtype", "float16"])
    
    print("=" * 65)
    print(f"🚀 Step 1/2: Starting vLLM Server with {VLLM_MODEL}...")
    print(f"Command: {' '.join(vllm_cmd)}")
    print("Backend: TORCH_SDPA (VLLM_USE_V1=0 for Tesla T4 compatibility)")
    print("=" * 65)
    
    # Redirect to log file to prevent OS pipe buffer deadlock (64KB limit)
    vllm_log = open("vllm.log", "w")
    vllm_proc = subprocess.Popen(vllm_cmd, env=vllm_env, stdout=vllm_log, stderr=subprocess.STDOUT)
    
    max_retries = 150  # 150 * 2s = 300s (5 minutes to allow download)
    vllm_ready = False
    print("\n⏳ Waiting for vLLM to download weights and initialize...")
    
    for i in range(max_retries):
        # Check if vLLM crashed
        if vllm_proc.poll() is not None:
            print(f"\n❌ vLLM process crashed with return code {vllm_proc.returncode}!")
            print("--- Last 30 lines of vllm.log ---")
            with open("vllm.log", "r") as f:
                print("".join(f.readlines()[-30:]))
            return False
        
        try:
            req = urllib.request.urlopen(f"{VLLM_URL}/v1/models", timeout=2)
            if req.status == 200:
                print(f"\n✅ vLLM server is READY! (took ~{(i+1)*2}s)")
                vllm_ready = True
                break
        except Exception:
            pass
        
        time.sleep(2)
        if (i + 1) % 5 == 0:
            last_line = ""
            if os.path.exists("vllm.log"):
                with open("vllm.log", "r") as f:
                    lines = [l.strip() for l in f.readlines() if l.strip()]
                    if lines:
                        last_line = lines[-1]
            print(f"  [{(i+1)*2}s / {max_retries*2}s] Latest log: {last_line[:85]}")
    
    if not vllm_ready:
        print("\n❌ vLLM timed out. Showing last 30 lines of vllm.log:")
        with open("vllm.log", "r") as f:
            print("".join(f.readlines()[-30:]))
        return False
    
    # 2. Start CLM Server (System One API)
    clm_cmd = [
        "clm-serve",
        "--port", str(CLM_PORT),
        "--emb-url", f"{VLLM_URL}/v1/embeddings",
        "--no-ui"
    ]
    
    print("\n" + "=" * 65)
    print("🚀 Step 2/2: Starting CLM Server (clm-serve)... ")
    print(f"Command: {' '.join(clm_cmd)}")
    print("=" * 65)
    
    clm_log = open("clm_serve.log", "w")
    clm_proc = subprocess.Popen(clm_cmd, stdout=clm_log, stderr=subprocess.STDOUT)
    
    clm_ready = False
    for i in range(30):
        if clm_proc.poll() is not None:
            print(f"\n❌ clm-serve crashed with code {clm_proc.returncode}!")
            with open("clm_serve.log", "r") as f:
                print("".join(f.readlines()[-30:]))
            return False
        try:
            req = urllib.request.urlopen(f"{CLM_URL}/health", timeout=2)
            if req.status == 200:
                print(f"\n✅ CLM Server is READY at {CLM_URL}!")
                clm_ready = True
                break
        except Exception:
            try:
                req = urllib.request.urlopen(f"{CLM_URL}/", timeout=2)
                print(f"\n✅ CLM Server is READY at {CLM_URL}!")
                clm_ready = True
                break
            except Exception:
                pass
        time.sleep(1)
    
    if not clm_ready:
        print("\n⚠️ Check clm_serve.log output:")
        with open("clm_serve.log", "r") as f:
            print("".join(f.readlines()[-20:]))
    
    return vllm_ready and clm_ready

start_services()


## 4. Initialize CLM Client

In [ ]:
from clm import CLMClient, Choice, Noul, Score

# Initialize CLM client connected to our local clm-serve instance
try:
    client = CLMClient(base_url=CLM_URL)
except TypeError:
    client = CLMClient()

print("✅ CLM Client initialized!")
print(f"Target CLM Server: {CLM_URL}")
print(f"Client type: {type(client)}")


## 5. Load Dataset

In [ ]:
# Dataset path - auto-detect for Google Colab / Local / Drive
candidate_paths = [
    "./eval/tool_call/datasets/multi_turn/batches/",
    "/content/fixed-uuid/eval/tool_call/datasets/multi_turn/batches/",
    "/content/eval/tool_call/datasets/multi_turn/batches/",
    "/content/batches/",
    "/content/drive/MyDrive/vinfast/jev_clm/batches/",
    "/home/admin123/Documents/fixed-uuid/eval/tool_call/datasets/multi_turn/batches/"
]

DATASET_PATH = next((p for p in candidate_paths if os.path.exists(p) and os.listdir(p)), candidate_paths[0])
print(f"📁 Using Dataset Path: {DATASET_PATH}")

def load_dataset(dataset_path: str, limit: Optional[int] = None) -> List[Dict]:
    """Load samples from JSON dataset files"""
    import glob
    
    all_samples = []
    json_files = sorted(glob.glob(os.path.join(dataset_path, "*.json")))
    
    if not json_files:
        print(f"⚠️ Warning: No .json files found in {dataset_path}!")
        print("Please upload batch json files or update DATASET_PATH.")
        return []
    
    for file_path in json_files:
        print(f"Loading: {file_path}")
        with open(file_path, 'r', encoding='utf-8') as f:
            samples = json.load(f)
            all_samples.extend(samples)
            print(f"  Loaded {len(samples)} samples")
    
    if limit:
        all_samples = all_samples[:limit]
    
    print(f"\nTotal samples loaded: {len(all_samples)}")
    return all_samples

# Load all samples
samples = load_dataset(DATASET_PATH)
if samples:
    print(f"\nFirst sample keys: {list(samples[0].keys())}")


## 6. Helper Functions

In [ ]:
# Candidate distractor tools from list_tool.json to make tool selection realistic
GLOBAL_DISTRACTOR_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Tra cứu thông tin dự báo thời tiết, nhiệt độ, lượng mưa, độ ẩm các tỉnh thành Việt Nam.",
            "parameters": {"type": "object", "properties": {"keyword": {"type": "string"}}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_gold_price",
            "description": "Tra cứu giá vàng thị trường thời gian thực hoặc xu hướng lịch sử giá vàng tại Việt Nam (SJC, PNJ, DOJI).",
            "parameters": {"type": "object", "properties": {"action": {"type": "string"}}}
        }
    }
]

def format_conversation(messages: List[Dict], max_turns: int = 3) -> str:
    """Format messages into a conversation string for CLM state (cutting off right at the user query before tool call)"""
    # 1. Tìm vị trí tool call cuối cùng của assistant để cắt ngữ cảnh đúng tại câu hỏi của User
    target_idx = len(messages)
    for idx in range(len(messages) - 1, -1, -1):
        if messages[idx].get("role") == "assistant" and messages[idx].get("tool_calls"):
            target_idx = idx
            break
            
    # Chỉ lấy các tin nhắn diễn ra TRƯỚC lượt gọi tool (tin nhắn cuối cùng sẽ là yêu cầu của User)
    relevant_messages = messages[:target_idx]
    
    # Bỏ qua system prompt
    relevant_messages = [m for m in relevant_messages if m.get("role") != "system"]
    if max_turns:
        relevant_messages = relevant_messages[-max_turns*2:]
        
    formatted_parts = []
    for msg in relevant_messages:
        role = msg.get("role", "user").capitalize()
        content = msg.get("content", "")
        if content:
            formatted_parts.append(f"{role}: {content}")
        elif msg.get("tool_calls"):
            for tc in msg["tool_calls"]:
                fname = tc.get("function", {}).get("name")
                formatted_parts.append(f"Assistant (called tool): {fname}")
        elif msg.get("tool_call_id"):
            preview = (msg.get("content", "") or "")[:100]
            formatted_parts.append(f"Tool: {preview}...")
            
    return "\n".join(formatted_parts)

def extract_tools_as_criteria(tools: List[Dict], include_distractors: bool = True) -> Dict[str, str]:
    """Convert tools to CLM Choice criteria format, optionally including distractor tools"""
    criteria = {}
    all_tools = list(tools)
    
    # Nếu dataset chỉ có 1 tool, thêm các tool đối chứng để CLM thực sự phải lựa chọn
    if include_distractors and len(all_tools) == 1:
        existing_names = {t.get("function", {}).get("name") for t in all_tools}
        for dt in GLOBAL_DISTRACTOR_TOOLS:
            if dt["function"]["name"] not in existing_names:
                all_tools.append(dt)
    
    for tool in all_tools:
        func = tool.get("function", tool)
        name = func.get("name", "")
        desc = func.get("description", "")
        params = func.get("parameters", {})
        if "properties" in params:
            param_names = list(params["properties"].keys())
            desc += f" (params: {", ".join(param_names)})"
        criteria[name] = desc
    return criteria

def extract_ground_truth(sample: Dict) -> Optional[Dict]:
    """Extract ground truth tool call from sample"""
    messages = sample.get("messages", [])
    
    for msg in reversed(messages):
        if msg.get("role") == "assistant" and msg.get("tool_calls"):
            tool_call = msg["tool_calls"][0]
            return {
                "tool_name": tool_call["function"]["name"],
                "arguments": tool_call["function"].get("arguments", "{}")
            }
    return None

print("✅ Helper functions defined (with user prompt cutoff & distractor tools)!")


## 7. CLM Inference Function

In [ ]:
def predict_tool_with_clm(
    client: CLMClient,
    messages: List[Dict],
    tools: List[Dict],
    max_turns: int = 3,
    include_distractors: bool = True
) -> Dict[str, Any]:
    """Predict tool call using CLM with proper SystemOneResponse parsing"""
    
    # Format conversation as state
    state = format_conversation(messages, max_turns)
    
    # Convert tools to criteria
    criteria = extract_tools_as_criteria(tools, include_distractors=include_distractors)
    
    if not criteria:
        return {"error": "No tools available", "predicted_tool": None}
    
    try:
        # Run CLM inference
        result = client.system_one(
            state=state,
            questions={
                "tool": Choice(
                    instructions="Select the most appropriate tool to call based on the conversation",
                    criteria=criteria
                )
            }
        )
        
        # Parse answer safely from SystemOneResponse
        # Response format: result.answers["tool"] -> ChoiceAnswer(choice="...", confidence=..., probabilities=...)
        tool_ans = result.answers.get("tool") if hasattr(result, "answers") else None
        
        if tool_ans is not None:
            predicted = getattr(tool_ans, "choice", None)
            prob = getattr(tool_ans, "confidence", 0.0)
            probs = getattr(tool_ans, "probabilities", {})
        else:
            # Fallback for dynamic attributes if SDK changes
            predicted = getattr(result, "tool", None)
            prob = getattr(result, "tool_prob", 0.0)
            probs = {}
        
        return {
            "predicted_tool": predicted,
            "probability": prob,
            "probabilities": probs,
            "all_criteria": list(criteria.keys()),
            "state_preview": state[:200] + "..." if len(state) > 200 else state
        }
        
    except Exception as e:
        return {"error": str(e), "predicted_tool": None}

print("✅ CLM inference function defined!")


## 8. Test Single Sample

In [ ]:
# Test with first sample
sample = samples[0]

print("=" * 60)
print(f"Sample ID: {sample['conversation_id']}")
print(f"Topic: {sample['conversation_topic']}")
print(f"Turn count: {sample['turn_count']}")
print("=" * 60)

# Show tools
print("\n📋 Available Tools:")
for tool in sample['tools']:
    func = tool['function']
    print(f"  - {func['name']}: {func['description'][:80]}...")

# Show conversation
print("\n💬 Conversation:")
for msg in sample['messages'][:6]:
    role = msg['role']
    content = msg.get('content', '')
    if content:
        print(f"  [{role}]: {content[:100]}...")
    elif msg.get('tool_calls'):
        for tc in msg['tool_calls']:
            print(f"  [{role}]: Call {tc['function']['name']}")

# Get ground truth
ground_truth = extract_ground_truth(sample)
print(f"\n✅ Ground Truth: {ground_truth}")

In [ ]:
# Run CLM prediction
print("\n" + "=" * 60)
print("🤖 Running CLM Inference...")
print("=" * 60)

start_time = time.time()
prediction = predict_tool_with_clm(client, sample['messages'], sample['tools'])
elapsed = time.time() - start_time

print(f"\n⏱️ Inference time: {elapsed:.2f}s")
print(f"\n🎯 Prediction: {prediction.get('predicted_tool')}")
print(f"📊 Probability: {prediction.get('probability', 'N/A')}")

# Check if correct
if ground_truth:
    predicted = prediction.get('predicted_tool')
    actual = ground_truth['tool_name']
    is_correct = predicted == actual
    print(f"\n{'✅ CORRECT!' if is_correct else '❌ INCORRECT'}")
    print(f"   Predicted: {predicted}")
    print(f"   Actual: {actual}")

## 9. Evaluate Multiple Samples

In [ ]:
def evaluate_samples(
    client: CLMClient,
    samples: List[Dict],
    max_samples: Optional[int] = None,
    include_distractors: bool = True,
    verbose: bool = True
) -> Dict[str, Any]:
    """Evaluate CLM on multiple samples with detailed breakdown"""
    
    eval_set = samples[:max_samples] if max_samples else samples
    total_to_run = len(eval_set)
    
    results = {
        "correct": 0,
        "incorrect": 0,
        "errors": 0,
        "total": 0,
        "latencies": [],
        "details": []
    }
    
    print(f"🚀 Starting evaluation on {total_to_run} samples...")
    
    for i, sample in enumerate(eval_set):
        ground_truth = extract_ground_truth(sample)
        
        if not ground_truth:
            results["errors"] += 1
            results["details"].append({
                "sample_id": sample.get("conversation_id", f"sample_{i}"),
                "error": "No ground truth tool_calls found in assistant messages"
            })
            continue
        
        # Run prediction
        start_time = time.time()
        prediction = predict_tool_with_clm(
            client,
            sample["messages"],
            sample["tools"],
            include_distractors=include_distractors
        )
        latency = time.time() - start_time
        
        results["latencies"].append(latency)
        results["total"] += 1
        
        # Check result
        if "error" in prediction and prediction["error"]:
            results["errors"] += 1
            if verbose and results["errors"] <= 3:
                print(f"  ⚠️ Error on sample {i+1}: {prediction['error']}")
            results["details"].append({
                "sample_id": sample.get("conversation_id", f"sample_{i}"),
                "error": prediction["error"],
                "correct": False
            })
        else:
            predicted = prediction.get("predicted_tool")
            actual = ground_truth["tool_name"]
            is_correct = (predicted == actual)
            
            if is_correct:
                results["correct"] += 1
            else:
                results["incorrect"] += 1
            
            results["details"].append({
                "sample_id": sample.get("conversation_id", f"sample_{i}"),
                "predicted": predicted,
                "actual": actual,
                "correct": is_correct,
                "probability": prediction.get("probability"),
                "all_criteria": prediction.get("all_criteria"),
                "latency": latency
            })
        
        if verbose and (i + 1) % 10 == 0:
            acc_so_far = (results["correct"] / results["total"]) * 100 if results["total"] > 0 else 0
            print(f"  Progress [{i+1}/{total_to_run}] -> Accuracy so far: {acc_so_far:.1f}% (Avg Latency: {sum(results['latencies'])/len(results['latencies']):.2f}s)")
    
    # Calculate metrics
    results["accuracy"] = results["correct"] / results["total"] if results["total"] > 0 else 0
    results["avg_latency"] = sum(results["latencies"]) / len(results["latencies"]) if results["latencies"] else 0
    
    return results

print("✅ Evaluation function defined!")


In [ ]:
# Run evaluation on 50 samples (hoặc toàn bộ dataset nếu len(samples) < 50)
EVAL_LIMIT = min(50, len(samples))

print("=" * 60)
print(f"🔬 Running Evaluation on {EVAL_LIMIT} samples (Total loaded: {len(samples)})")
print("=" * 60)

results = evaluate_samples(client, samples, max_samples=EVAL_LIMIT, include_distractors=True, verbose=True)

print("\n" + "=" * 60)
print("📊 RESULTS SUMMARY")
print("=" * 60)
print(f"Total Evaluated: {results['total']}")
print(f"✅ Correct:       {results['correct']}")
print(f"❌ Incorrect:     {results['incorrect']}")
print(f"⚠️ Errors:        {results['errors']}")
print(f"🎯 Accuracy:      {results['accuracy']*100:.1f}%")
print(f"⏱️ Avg Latency:   {results['avg_latency']:.3f}s ({results['avg_latency']*1000:.1f}ms)")
if results["latencies"]:
    print(f"⏱️ Min Latency:   {min(results['latencies']):.3f}s")
    print(f"⏱️ Max Latency:   {max(results['latencies']):.3f}s")

## 10. Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Accuracy pie chart
ax1 = axes[0]
labels = ['Correct', 'Incorrect']
sizes = [results['correct'], results['incorrect']]
colors = ['#2ecc71', '#e74c3c']
ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title('CLM Accuracy')

# 2. Latency distribution
ax2 = axes[1]
ax2.hist(results['latencies'], bins=10, color='#3498db', edgecolor='black', alpha=0.7)
ax2.axvline(results['avg_latency'], color='red', linestyle='--', label=f'Avg: {results["avg_latency"]:.2f}s')
ax2.set_xlabel('Latency (seconds)')
ax2.set_ylabel('Count')
ax2.set_title('Latency Distribution')
ax2.legend()

# 3. Results bar chart
ax3 = axes[2]
categories = ['Correct', 'Incorrect', 'Errors']
values = [results['correct'], results['incorrect'], results['errors']]
colors = ['#2ecc71', '#e74c3c', '#f39c12']
ax3.bar(categories, values, color=colors)
ax3.set_ylabel('Count')
ax3.set_title('Prediction Results')
for i, v in enumerate(values):
    ax3.text(i, v + 0.1, str(v), ha='center')

plt.tight_layout()
plt.show()

## 11. Show Sample Results

In [ ]:
# Show detailed results for first 10 samples
print("=" * 80)
print("📋 DETAILED RESULTS (First 10 samples)")
print("=" * 80)

for i, detail in enumerate(results['details'][:10]):
    status = "✅" if detail.get("correct") else "❌"
    print(f"\n{i+1}. Sample ID: {detail.get('sample_id')}")
    print(f"   {status} Predicted: {detail.get('predicted', 'N/A')} | Actual: {detail.get('actual', 'N/A')}")
    if detail.get("probability") is not None:
        print(f"   📊 Confidence: {detail['probability']:.3f}")
    if detail.get("latency"):
        print(f"   ⏱️ Latency:    {detail['latency']:.3f}s")
    if detail.get("error"):
        print(f"   ⚠️ Error msg:   {detail['error']}")


## 12. Full Evaluation (More Samples)

In [ ]:
# Run on more samples (50)
print("=" * 60)
print("🔬 Full Evaluation (50 samples)")
print("=" * 60)

full_results = evaluate_samples(client, samples, max_samples=50, verbose=True)

print("\n" + "=" * 60)
print("📊 FULL RESULTS SUMMARY")
print("=" * 60)
print(f"Total samples: {full_results['total']}")
print(f"Correct: {full_results['correct']}")
print(f"Incorrect: {full_results['incorrect']}")
print(f"Errors: {full_results['errors']}")
print(f"\n🎯 Accuracy: {full_results['accuracy']*100:.1f}%")
print(f"⏱️ Avg latency: {full_results['avg_latency']:.2f}s")
print(f"⏱️ Min latency: {min(full_results['latencies']):.2f}s")
print(f"⏱️ Max latency: {max(full_results['latencies']):.2f}s")

## 13. Compare with Different Max Turns

In [ ]:
# Test with different context window sizes
def predict_with_config(
    messages: List[Dict],
    tools: List[Dict],
    max_turns: int
) -> Dict:
    """Predict with configurable max turns"""
    state = format_conversation(messages, max_turns)
    criteria = extract_tools_as_criteria(tools, include_distractors=True)
    
    if not criteria:
        return {"error": "No tools"}
    
    try:
        result = client.system_one(
            state=state,
            questions={
                "tool": Choice(
                    instructions="Select the appropriate tool",
                    criteria=criteria
                )
            }
        )
        tool_ans = result.answers.get("tool") if hasattr(result, "answers") else None
        if tool_ans:
            return {"predicted": tool_ans.choice, "prob": tool_ans.confidence}
        return {"predicted": getattr(result, "tool", None), "prob": getattr(result, "tool_prob", 0.0)}
    except Exception as e:
        return {"error": str(e)}

# Test different max_turns
print("Testing different context window sizes...")
print("-" * 60)

turns_to_test = [1, 2, 3, 5]
turns_results = {}

for max_turns in turns_to_test:
    correct = 0
    total = 0
    
    for sample in samples[:30]:
        ground_truth = extract_ground_truth(sample)
        if not ground_truth:
            continue
        
        prediction = predict_with_config(
            sample["messages"],
            sample["tools"],
            max_turns
        )
        
        if "error" not in prediction or not prediction["error"]:
            total += 1
            if prediction.get("predicted") == ground_truth["tool_name"]:
                correct += 1
    
    accuracy = correct / total if total > 0 else 0
    turns_results[max_turns] = accuracy
    print(f"Max turns: {max_turns} -> Accuracy: {accuracy*100:.1f}%")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(list(turns_results.keys()), [v*100 for v in turns_results.values()], "bo-")
plt.xlabel("Max Turns")
plt.ylabel("Accuracy (%)")
plt.title("CLM Accuracy vs Context Window Size")
plt.xticks(list(turns_results.keys()))
plt.ylim(0, 100)
plt.grid(True, alpha=0.3)
plt.show()


## 14. Interactive Demo

In [ ]:
# Interactive demo - try different samples
def demo_sample(sample_idx: int):
    """Demo a specific sample"""
    sample = samples[sample_idx]
    
    print("=" * 70)
    print(f"SAMPLE {sample_idx}: {sample['conversation_id']}")
    print("=" * 70)
    
    # Show topic
    print(f"\n📌 Topic: {sample['conversation_topic']}")
    
    # Show tools
    print("\n🔧 Available Tools:")
    for tool in sample['tools']:
        print(f"   - {tool['function']['name']}")
    
    # Show conversation
    print("\n💬 Conversation:")
    for msg in sample['messages']:
        if msg['role'] != 'system':
            content = msg.get('content', '')
            if content:
                print(f"   [{msg['role']}]: {content[:80]}...")
    
    # Get ground truth
    ground_truth = extract_ground_truth(sample)
    print(f"\n✅ Ground Truth: {ground_truth['tool_name'] if ground_truth else 'N/A'}")
    
    # Run prediction
    start = time.time()
    prediction = predict_tool_with_clm(client, sample['messages'], sample['tools'])
    elapsed = time.time() - start
    
    print(f"\n🤖 CLM Prediction: {prediction.get('predicted_tool')}")
    print(f"📊 Probability: {prediction.get('probability', 'N/A'):.3f}")
    print(f"⏱️ Latency: {elapsed:.2f}s")
    
    if ground_truth:
        is_correct = prediction.get('predicted_tool') == ground_truth['tool_name']
        print(f"\n{'✅ CORRECT!' if is_correct else '❌ INCORRECT'}")
    
    return prediction

# Try sample 0
demo_sample(0)

In [ ]:
# Try more samples interactively
# Change the index to try different samples
demo_sample(5)

In [ ]:
demo_sample(10)

## 15. Export Results

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Create output directory structure
OUTPUT_BASE = "/content/drive/MyDrive/vinfast/jev_clm/results"
CLM_OUTPUT_DIR = f"{OUTPUT_BASE}/clm"
JEV_OUTPUT_DIR = f"{OUTPUT_BASE}/jev"

# Create directories
os.makedirs(CLM_OUTPUT_DIR, exist_ok=True)
os.makedirs(JEV_OUTPUT_DIR, exist_ok=True)

print(f"✅ Directories created:")
print(f"   - {CLM_OUTPUT_DIR}")
print(f"   - {JEV_OUTPUT_DIR}")

# Export full results to JSON
export_data = {
    'model': 'CLM-v0.1-8B',
    'evaluation_date': time.strftime('%Y-%m-%d %H:%M:%S'),
    'total_samples': full_results['total'],
    'correct': full_results['correct'],
    'incorrect': full_results['incorrect'],
    'errors': full_results['errors'],
    'accuracy': full_results['accuracy'],
    'avg_latency': full_results['avg_latency'],
    'min_latency': min(full_results['latencies']),
    'max_latency': max(full_results['latencies']),
    'details': full_results['details']
}

# Save to Google Drive
output_file = f"{CLM_OUTPUT_DIR}/clm_results_{time.strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"\n✅ Results saved to: {output_file}")
print("\n" + "=" * 60)
print("📄 EXPORTED DATA PREVIEW")
print("=" * 60)
print(json.dumps({k: v for k, v in export_data.items() if k != 'details'}, indent=2))

---

## Summary

This notebook demonstrates:

1. **Installation** - CLM and vLLM setup
2. **Dataset Loading** - Multi-turn function calling dataset
3. **CLM Inference** - Predict tool calls using CLM
4. **Evaluation** - Accuracy and latency metrics
5. **Visualization** - Results charts
6. **Comparison** - Different context window sizes

**Note:** CLM achieves strong performance on tool selection while being significantly faster than traditional generation-based models.

---

For more information:
- **CLM Model:** https://huggingface.co/Contrastive-LM/CLM-v0.1-8B
- **CLM GitHub:** https://github.com/Contrastive-LM/CLM
- **Documentation:** See `clm_detailed.md`